# 04 — Mark 4 — Two-Stage Multi-Window Validation Smoke

[![Phase](https://img.shields.io/badge/Pipeline-Mark%201%20to%204E-blue.svg)]()
[![Mode](https://img.shields.io/badge/Default-REUSE%20(fast%2C%20deterministic)-success.svg)]()

**Pipeline position:** notebook **04 of 09** — run the suite in order 00 → 09.
**Original:** `mark 1/mark_4_two_stage_validation_smoke.ipynb`

## Objective

Warm-started 5-epoch smoke on the locked training split: does the full two-stage pipeline (ROI crop → multi-window → tumour head) train stably and hit the continuation targets on validation?

## Inputs (read-only)

- `mark_3_gate_result.json`, `training_roi_manifest.csv` / `validation_roi_manifest.csv`
- Warm-start checkpoint (multi-task epoch-8) for the ROI stage
- REUSE: archived `mark 1/mark_4_outputs/mark_4_history.csv` + best checkpoint; REBUILD: retrain (RUN_MARK4_SMOKE=True, REUSE_HISTORY=False)

## Outputs → `Evaluation/mark_1_to_4e_outputs/mark_4_outputs/`

Every file below keeps the exact naming used by the archived run, so results are
directly comparable with the original `mark 1/mark_*_outputs/` outputs.

| File |
|---|
| `mark_4_gate_result.json` |
| `mark_4_history.csv` |
| `training_roi_manifest.csv` |
| `validation_roi_manifest.csv` |
| `roi_dataset_audit.png` |
| `roi_patient_results.csv` |
| `best_validation_patient_metrics.csv` |
| `best_validation_per_slice.csv` |
| `best_validation_size_metrics.csv` |
| `expected_vs_actual.csv` |
| `broad_png_source_parity.csv` |
| `mark_4_smoke_dashboard.png` |
| `best_full_image_predictions.png` |

**Visualizations produced by this notebook:** `roi_dataset_audit.png`, `mark_4_smoke_dashboard.png`, `best_full_image_predictions.png`

## Phase dataflow

```mermaid
flowchart LR
  A["inputs: mark_3_gate_result.json, Warm-start checkpoint (multi-task epoch-8) for the ROI stage, mark 1/mark_4_outputs/mark_4_history.csv"] -->
  B[phase cells: provenance + reuse/rebuild + compute]
  B --> G["gate: mark_4_gate_result.json"]
  B --> O[organized per-phase outputs]
  G --> D[downstream notebook reads this gate]
```


## Key finding (reproduced)

**Smoke test PASSED with caveats**: training is stable and the pipeline is sound, but only **5/6 continuation targets** are met at the frozen threshold (recall on small lesions remains the bottleneck).

## Gate

`mark_4_gate_result.json` — continuation targets (5 of 6 met in the archived run)

## Run notes

5 epochs, warm-started, validation-only evaluation. Test split locked.

> **Shared setup:** the next cell is the *identical* global-setup cell embedded in every notebook
> (paths, seeds, provenance hashes, test lock, shared helpers). REUSE mode reads frozen artifacts from
> `mark 1/`, so each notebook is deterministic and reproducible; set the `REUSE_*` / `RUN_*` flags to
> rebuild caches or retrain (GPU hours).
>
> **Ordering matters:** this phase reads the previous phase's gate JSON from the shared output folder
> (`mark_1_to_4e_outputs/…`), so run the suite in order **00 → 09**. A phase can be re-run standalone
> once its upstream gates exist (re-running the preceding notebooks regenerates them).

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
# Centralized shared output root under Evaluation/output (one folder per notebook)
SHARED_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "output"
# Legacy outputs (read-only fallback for the availability-check import helpers)
LEGACY_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
PHASE_DIR = {
    "00_setup": "00_pipeline_overview",
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
OUT       = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "data"    for phase in PHASE_DIR}
OUT_FIGS  = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "figures" for phase in PHASE_DIR}
OUT_CACHE = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "caches"  for phase in PHASE_DIR}
CONSOLIDATED = OUT["consolidated"]
CONSOLIDATED_FIGS = OUT_FIGS["consolidated"]
for _d in [*OUT.values(), *OUT_FIGS.values(), *OUT_CACHE.values()]:
    _d.mkdir(parents=True, exist_ok=True)
NOTEBOOK_KEY = "mark_4"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Shared output helpers: availability check, cross-notebook import, registry
# ---------------------------------------------------------------------------
import shutil as _shutil

ARTIFACT_INDEX = SHARED_OUTPUT_ROOT / "artifact_index.json"


def _load_artifact_index():
    if ARTIFACT_INDEX.is_file():
        return json.loads(ARTIFACT_INDEX.read_text())
    return {"version": 1, "artifacts": []}


def _save_artifact_index(index):
    ARTIFACT_INDEX.write_text(json.dumps(index, indent=2))


def register_artifact(name, kind="data", phase=None):
    phase = phase or NOTEBOOK_KEY
    index = _load_artifact_index()
    index["artifacts"] = [a for a in index["artifacts"]
                          if not (a.get("phase") == phase and a.get("name") == name)]
    path = OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name
    index["artifacts"].append({
        "phase": phase, "name": name, "kind": kind,
        "sha256": sha256_file(path) if path.is_file() else None,
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    })
    _save_artifact_index(index)


def shared_path(phase, name, kind="data"):
    return OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name


def legacy_path(phase, name, kind="data"):
    return LEGACY_OUTPUT_ROOT / f"{phase}_outputs" / name


def load_shared(phase, name, kind="data", required=True):
    """Availability check: Evaluation/output -> legacy mark_1_to_4e_outputs -> compute/raise."""
    target = shared_path(phase, name, kind)
    if target.is_file():
        return target
    legacy = legacy_path(phase, name, kind)
    if legacy.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy2(legacy, target)
        print(f"IMPORT: reused legacy {phase}/{name} (copied to {target}).")
        return target
    if required:
        raise FileNotFoundError(
            f"Required upstream output missing: {PHASE_DIR.get(phase, phase)}/{name}.\n"
            f"Run the notebook for phase '{phase}' first (outputs land under "
            f"{SHARED_OUTPUT_ROOT / PHASE_DIR.get(phase, phase)}).")
    return None


def require_upstream_gate(phase, gate_name=None):
    gate_name = gate_name or f"{phase}_gate_result.json"
    return json.loads(load_shared(phase, gate_name, "data", required=True).read_text())


def save_figure(fig, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT_FIGS[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=170, bbox_inches="tight")
    register_artifact(name, "figures", phase)
    return target


def save_table(frame, name, phase=None, index=False):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=index)
    register_artifact(name, "data", phase)
    return target


def save_json(obj, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(obj, indent=2))
    register_artifact(name, "data", phase)
    return target


# ---------------------------------------------------------------------------
# Standardized per-phase summary dashboard
# ---------------------------------------------------------------------------
CORE_METRICS = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct",
                "empty_slice_false_positive_pct"]
TARGETS_SHEET = {phase: CONTINUATION_TARGETS for phase in
                 ["mark_1", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
GATE_SELECTOR = {
    "mark_1": "best_observed_configuration_for_diagnosis",
    "mark_2": "selected_roi_configuration",
    "mark_3": "selected_configuration",
    "mark_4": "best_metrics", "mark_4b": "selected_metrics",
    "mark_4c": "arms", "mark_4d": "selected_metrics", "mark_4e": "selected_metrics",
}
TREND_CSV = {
    "mark_1":  ("calibration_configuration_results.csv", "tumor_threshold"),
    "mark_2":  ("roi_configuration_results.csv", "liver_threshold"),
    "mark_3":  ("overfit_history.csv", "epoch"),
    "mark_4":  ("mark_4_history.csv", "epoch"),
    "mark_4b": ("threshold_results.csv", "threshold"),
    "mark_4c": ("mark_4c_history.csv", "epoch"),
    "mark_4d": ("reconciled_threshold_results.csv", "threshold"),
    "mark_4e": ("fusion_threshold_results.csv", "threshold"),
}


def _metric_color(metric, value, targets):
    if not targets or metric not in targets:
        return "#4C72B0"
    target = targets[metric]
    passed = (value <= target if metric in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else value >= target)
    return "#2E9E5B" if passed else "#C44E52"


def render_summary_dashboard(phase):
    gate = json.loads((OUT[phase] / f"{phase}_gate_result.json").read_text())
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes[0, 0].axis("off")
    text_lines = [f"phase: {phase}", f"status: {gate.get('status')}",
                  "decision: %s" % (gate.get("decision") or gate.get("next_notebook")
                                    or gate.get("next_step") or "-")]
    for key in ("test_images_accessed", "manifest_sha256", "next_mark", "next_step"):
        if key in gate:
            text_lines.append(f"{key}: {gate[key]}")
    axes[0, 0].text(0.02, 0.99, "\n".join(text_lines), transform=axes[0, 0].transAxes,
                    va="top", ha="left", fontsize=9, family="monospace")
    axes[0, 0].set_title("Gate metadata", fontsize=11, weight="bold")

    selector = GATE_SELECTOR.get(phase)
    selected = gate.get(selector) if selector else None
    targets = TARGETS_SHEET.get(phase)
    row = None
    if isinstance(selected, list):
        chosen = gate.get("selected_arm") or (selected[0].get("arm") if selected else None)
        for arm in selected:
            if arm.get("arm") == chosen:
                row = arm
    else:
        row = selected
    axes[1, 0].set_title("Selected metrics vs targets (green=pass, red=miss)",
                         fontsize=10, weight="bold")
    if row is not None:
        metric_names = [m for m in CORE_METRICS if m in row]
        if metric_names:
            values = [float(row[m]) for m in metric_names]
            axes[1, 0].bar(np.arange(len(metric_names)), values,
                           color=[_metric_color(m, float(row[m]), targets) for m in metric_names])
            axes[1, 0].axhline(0, color="k", lw=0.8)
            for metric in metric_names:
                if targets and metric in targets:
                    axes[1, 0].axhline(targets[metric], color="gray", lw=0.8, ls="--")
            axes[1, 0].set_xticks(np.arange(len(metric_names)))
            axes[1, 0].set_xticklabels(metric_names, rotation=30, ha="right", fontsize=8)
            axes[1, 0].set_ylabel("value")
            if isinstance(selected, list) and row.get("arm"):
                axes[1, 0].set_title(f"Selected arm: {row['arm']} vs targets",
                                     fontsize=10, weight="bold")
        else:
            axes[1, 0].axis("off")
            axes[1, 0].text(0.5, 0.5, "No core-metric table in gate selector",
                            ha="center", va="center")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.5, 0.5, "No selector in gate JSON", ha="center", va="center")

    csv_name, x_col = TREND_CSV.get(phase, (None, None))
    trend_path = (OUT[phase] / csv_name) if csv_name else None
    if trend_path is not None and trend_path.is_file():
        trend = pd.read_csv(trend_path)
        axes[1, 1].set_title(f"Trend: {csv_name} (x={x_col})", fontsize=10, weight="bold")
        if phase in ("mark_3", "mark_4c"):
            group_col = "configuration" if phase == "mark_3" else "arm"
            y_col = "hard_micro_dice" if phase == "mark_3" else "mean_patient_dice"
            for label, group in trend.groupby(group_col):
                axes[1, 1].plot(group[x_col], group[y_col], marker="o", ms=3, label=str(label))
            axes[1, 1].legend(fontsize=7)
        else:
            y_col = "mean_patient_dice" if "mean_patient_dice" in trend.columns else trend.columns[1]
            axes[1, 1].plot(trend[x_col], trend[y_col], marker="o", ms=3, color="#4C72B0")
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel("metric")
    else:
        axes[1, 1].axis("off")
        axes[1, 1].text(0.5, 0.5, "Trend CSV not available yet - compute the phase first",
                        ha="center", va="center")

    axes[0, 1].axis("off")
    produced = sorted(p.name for p in OUT[phase].iterdir() if p.is_file())
    inventory = "\n".join(f"- {name}" for name in produced[:20])
    axes[0, 1].text(0.02, 0.99, inventory or "(no data artifacts yet)",
                    transform=axes[0, 1].transAxes, va="top", ha="left", fontsize=8,
                    family="monospace")
    axes[0, 1].set_title(f"Produced artifacts (Evaluation/output/{PHASE_DIR[phase]}/data)",
                         fontsize=10, weight="bold")

    figure.suptitle(f"{phase} - phase summary dashboard", fontsize=15, weight="bold")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    save_figure(figure, f"{phase}_summary_dashboard.png", phase=phase)
    plt.show()
    print(f"PASS: {phase}_summary_dashboard.png -> {OUT_FIGS[phase]}")

# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {SHARED_OUTPUT_ROOT}")

Device: cuda | REUSE_CACHES=True | REUSE_HISTORY=True
PASS: provenance, split geometry, and test lock verified.
Outputs: D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output


# Part 4 — Mark 4: Two-Stage ROI Validation Smoke Test

**Original:** `mark 1/mark_4_two_stage_validation_smoke.ipynb`

## Question

After the 16-slice overfit proof, does the same frozen pipeline generalize across **all** training and
validation patients in a bounded 5-epoch smoke run?

## Key finding (reproduced)

- **Smoke gate: FAILED for continuation** (5/6 temporary targets at best epoch 5).
- Best epoch 5: mean patient Dice 0.3639, V104 0.0672, V116 0.0108, Q1 42.59%,
  **positive predicted-empty 36.85%** (> 35% target), empty-slice FP 3.42%.
- The recall failure is concentrated on tumor-positive slices → Mark 4B probes threshold calibration.

## Contract

- Frozen pipeline: ROI (0.50, `largest_3d`, pad 16) + broad window `[-160,240]`, tumor threshold 0.50.
- Fresh warm start from the epoch-8 multi-task checkpoint (the 16-slice memorization checkpoint is NOT used).
- Temporary continuation targets: mean patient Dice ≥ 0.3329, V104 ≥ 0.05, V116 ≥ 0.01,
  Q1 ≥ 35%, positive empty ≤ 35%, empty FP ≤ 20%.
- Test split locked.

### 4.1 Freeze training + validation ROI manifests

In [2]:
import shutil
from src.framework.losses.focal_dice import FocalDiceLoss
from src.framework.models.mobilenetv2_unet import MobileNetV2UNet

training_rois = pd.read_csv(load_shared("mark_3", "training_roi_manifest.csv"))
assert len(training_rois) == 104 and not training_rois["roi_empty"].astype(bool).any()

mark2_roi_rows = pd.read_csv(load_shared("mark_2", "roi_patient_results.csv"))
validation_rois = mark2_roi_rows.loc[
    mark2_roi_rows["liver_threshold"].eq(0.50)
    & mark2_roi_rows["padding"].eq(16)
    & mark2_roi_rows["component_mode"].eq("largest_3d")].copy()
assert len(validation_rois) == 13 and not validation_rois["roi_empty"].astype(bool).any()
validation_rois.to_csv(OUT["mark_4"] / "validation_roi_manifest.csv", index=False)

roi_summary = pd.DataFrame([
    {"split": "train", "patients": len(training_rois),
     "median_area": training_rois["crop_area_ratio"].median(),
     "max_area": training_rois["crop_area_ratio"].max(),
     "empty_rois": int(training_rois["roi_empty"].sum())},
    {"split": "validation", "patients": len(validation_rois),
     "median_area": validation_rois["crop_area_ratio"].median(),
     "max_area": validation_rois["crop_area_ratio"].max(),
     "empty_rois": int(validation_rois["roi_empty"].sum())},
])
display(roi_summary)

,split,patients,median_area,max_area,empty_rois
0,train,104,0.420425,0.606201,0
1,validation,13,0.427002,0.533203,0


### 4.2 Build the ROI dataset and audit PNG/source-HU parity

In [3]:
import cv2


class SynchronizedROIAugment:
    def __call__(self, image, mask):
        if random.random() < 0.30:
            image, mask = np.fliplr(image).copy(), np.fliplr(mask).copy()
        if random.random() < 0.50:
            height, width = image.shape
            matrix = cv2.getRotationMatrix2D(
                (width / 2, height / 2), random.uniform(-8, 8),
                random.uniform(0.97, 1.03))
            image = cv2.warpAffine(image, matrix, (width, height), flags=cv2.INTER_LINEAR)
            mask = cv2.warpAffine(mask.astype(np.uint8), matrix, (width, height),
                                  flags=cv2.INTER_NEAREST) > 0
        if random.random() < 0.35:
            image = np.clip(image + np.random.normal(0, 0.015, image.shape), 0, 1)
        return image.astype(np.float32), mask.astype(np.float32)


class ROISliceDataset(Dataset):
    def __init__(self, rows, roi_frame, augment=None):
        self.rows = rows.reset_index(drop=True)
        self.rois = roi_frame.set_index("volume_id")
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        box = self.rois.loc[int(row.volume_id)]
        y0, y1, x0, x1 = [int(box[key]) for key in ("y0", "y1", "x0", "x1")]
        with Image.open(DATASET_ROOT / row.image_path) as handle:
            image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
        with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
            truth_full = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
        image_roi = resize_float(image[y0:y1, x0:x1])
        truth_roi = resize_mask(truth_full[y0:y1, x0:x1])
        if self.augment is not None:
            image_roi, truth_roi = self.augment(image_roi, truth_roi)
        return {"image": torch.from_numpy(image_roi[None]).float(),
                "mask": torch.from_numpy(np.asarray(truth_roi)[None]).float(),
                "sample_id": str(row.sample_id), "volume_id": int(row.volume_id),
                "slice_index": int(row.slice_index),
                "true_pixels_full": int(row.tumor_pixels),
                "box": torch.tensor([y0, y1, x0, x1], dtype=torch.int32)}


train_dataset = ROISliceDataset(train_manifest, training_rois,
                                augment=SynchronizedROIAugment())
validation_dataset = ROISliceDataset(validation_manifest, validation_rois, augment=None)
print(f"Train={len(train_dataset):,} | Validation={len(validation_dataset):,}")

import nibabel as nib
audit_rows = train_manifest.sample(32, random_state=SEED)
parity_rows, loaded_volumes = [], {}
for row in audit_rows.itertuples(index=False):
    if row.source_volume_path not in loaded_volumes:
        loaded_volumes[row.source_volume_path] = nib.load(str(row.source_volume_path))
    hu = np.asanyarray(
        loaded_volumes[row.source_volume_path].dataobj[:, :, int(row.slice_index)]
    ).astype(np.float32)
    regenerated = resize_float(np.clip((hu - BROAD_WINDOW[0]) / (BROAD_WINDOW[1] - BROAD_WINDOW[0]), 0, 1))
    with Image.open(DATASET_ROOT / row.image_path) as handle:
        stored = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
    parity_rows.append({"sample_id": row.sample_id,
                        "mean_absolute_error": float(np.mean(np.abs(regenerated - stored))),
                        "max_absolute_error": float(np.max(np.abs(regenerated - stored)))})
parity = pd.DataFrame(parity_rows)
parity.to_csv(OUT["mark_4"] / "broad_png_source_parity.csv", index=False)
print(parity.describe())
assert parity["mean_absolute_error"].median() <= 0.01

preview = [train_dataset[i] for i in [0, 1000, 10000, 30000]]
figure, axes = plt.subplots(4, 2, figsize=(9, 16))
for row_axes, item in zip(axes, preview):
    row_axes[0].imshow(item["image"][0], cmap="gray", vmin=0, vmax=1)
    row_axes[0].set_title(f"{item['sample_id']} broad ROI")
    row_axes[1].imshow(item["mask"][0], cmap="gray", vmin=0, vmax=1)
    row_axes[1].set_title("Tumor target")
    for axis in row_axes:
        axis.axis("off")
figure.suptitle("Full-dataset ROI geometry audit", fontsize=16, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4"] / "roi_dataset_audit.png", dpi=170, bbox_inches="tight")
plt.show()

Train=40,667 | Validation=10,685
       mean_absolute_error  max_absolute_error
count            32.000000           32.000000
mean              0.000529            0.004485
std               0.000124            0.000149
min               0.000283            0.004277
25%               0.000443            0.004367
50%               0.000539            0.004465
75%               0.000599            0.004581
max               0.000744            0.004874


C:\Users\alanm\AppData\Local\Temp\ipykernel_26868\1170923708.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.3 Patient-aware stratified loaders + fresh warm start

In [4]:
volume_counts = train_manifest["volume_id"].value_counts()
weights = train_manifest["volume_id"].map(
    lambda v: 1.0 / volume_counts.loc[v]).to_numpy(dtype=np.float64)
weights *= np.where(train_manifest["tumor_pixels"].to_numpy() > 0, 3.0, 1.0)
weights /= weights.mean()
sampler_generator = torch.Generator().manual_seed(SEED)
sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double),
                                num_samples=len(train_dataset), replacement=True,
                                generator=sampler_generator)
train_loader = DataLoader(train_dataset, batch_size=4, sampler=sampler,
                          num_workers=0, pin_memory=torch.cuda.is_available())
validation_loader = DataLoader(validation_dataset, batch_size=8, shuffle=False,
                               num_workers=0, pin_memory=torch.cuda.is_available())

source_payload = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)
source_state = source_payload["model_state"]
model = MobileNetV2UNet(in_channels=1, out_channels=1, pretrained=False)
target_state = model.state_dict()
for key, value in source_state.items():
    if key in target_state and target_state[key].shape == value.shape:
        target_state[key] = value.clone()
target_state["final.weight"] = source_state["final.weight"][1:2].clone()
target_state["final.bias"] = source_state["final.bias"][1:2].clone()
model.load_state_dict(target_state, strict=True)
model.to(DEVICE)

loss_function = FocalDiceLoss(focal_alpha=0.75, focal_gamma=2.0,
                              focal_weight=0.5, dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5, eta_min=3e-5)
print("PASS: fresh warm start from epoch-8 multi-task checkpoint.")

PASS: fresh warm start from epoch-8 multi-task checkpoint.


### 4.4 Full-image validation metrics (ROI → 256×256 inverse mapping)

In [5]:
def evaluate(model):
    model.eval()
    patient_accumulators, slice_rows, validation_loss = {}, [], 0.0
    with torch.inference_mode():
        for batch in validation_loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            masks = batch["mask"].to(DEVICE, non_blocking=True)
            logits = model(images)
            validation_loss += float(loss_function(logits, masks)) * len(images)
            probabilities = torch.sigmoid(logits).cpu().numpy()[:, 0]
            for index, sample_id in enumerate(batch["sample_id"]):
                volume_id = int(batch["volume_id"][index])
                slice_index = int(batch["slice_index"][index])
                box = batch["box"][index].numpy()
                probability_full = probability_to_full(probabilities[index], box)
                prediction = probability_full >= 0.50
                manifest_row = validation_manifest.loc[
                    validation_manifest["sample_id"].eq(sample_id)].iloc[0]
                with Image.open(DATASET_ROOT / manifest_row["tumor_mask_path"]) as handle:
                    truth = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
                intersection = int((prediction & truth).sum())
                predicted, true = int(prediction.sum()), int(truth.sum())
                values = patient_accumulators.setdefault(volume_id, {
                    "intersection": 0, "predicted": 0, "true": 0,
                    "positive": 0, "detected": 0, "positive_empty": 0,
                    "empty": 0, "empty_fp": 0})
                values["intersection"] += intersection
                values["predicted"] += predicted
                values["true"] += true
                if true > 0:
                    values["positive"] += 1
                    values["detected"] += int(intersection > 0)
                    values["positive_empty"] += int(predicted == 0)
                else:
                    values["empty"] += 1
                    values["empty_fp"] += int(predicted > 0)
                slice_rows.append({"sample_id": sample_id, "volume_id": volume_id,
                                   "slice_index": slice_index, "true_pixels": true,
                                   "predicted_pixels": predicted,
                                   "intersection_pixels": intersection,
                                   "dice": (2 * intersection + 1e-6) / (predicted + true + 1e-6)})
    patient_rows = [{"volume_id": v, "true_pixels": x["true"],
                     "predicted_pixels": x["predicted"],
                     "micro_dice": (2 * x["intersection"] + 1e-6) / (x["predicted"] + x["true"] + 1e-6),
                     "positive_predicted_empty_pct": 100 * x["positive_empty"] / max(x["positive"], 1),
                     "empty_slice_false_positive_pct": 100 * x["empty_fp"] / max(x["empty"], 1)}
                    for v, x in sorted(patient_accumulators.items())]
    patients = pd.DataFrame(patient_rows)
    slices = pd.DataFrame(slice_rows)
    positive_patients = patients.loc[patients["true_pixels"].gt(0)]
    positive_slices = slices.loc[slices["true_pixels"].gt(0)].copy()
    positive_slices["size_quartile"] = pd.qcut(positive_slices["true_pixels"], 4,
                                               labels=["Q1", "Q2", "Q3", "Q4"])
    size_rows = [{"size_quartile": str(q), "slices": len(g),
                  "mean_dice": g["dice"].mean(),
                  "detected_pct": 100 * (g["intersection_pixels"] > 0).mean(),
                  "predicted_empty_pct": 100 * (g["predicted_pixels"] == 0).mean()}
                 for q, g in positive_slices.groupby("size_quartile", observed=True)]
    sizes = pd.DataFrame(size_rows)
    totals = patient_accumulators.values()
    result = {
        "validation_loss": validation_loss / len(validation_dataset),
        "mean_patient_dice": float(positive_patients["micro_dice"].mean()),
        "median_patient_dice": float(positive_patients["micro_dice"].median()),
        "worst_patient_dice": float(positive_patients["micro_dice"].min()),
        "volume_104_dice": float(patients.set_index("volume_id")["micro_dice"].get(104, np.nan)),
        "volume_116_dice": float(patients.set_index("volume_id")["micro_dice"].get(116, np.nan)),
        "q1_detected_pct": float(sizes.set_index("size_quartile").loc["Q1", "detected_pct"]),
        "positive_predicted_empty_pct": float(100 * sum(v["positive_empty"] for v in totals)
                                              / max(sum(v["positive"] for v in totals), 1)),
        "empty_slice_false_positive_pct": float(100 * sum(v["empty_fp"] for v in totals)
                                                / max(sum(v["empty"] for v in totals), 1)),
    }
    return result, patients, slices, sizes

### 4.5 Smoke training (reuse frozen history or retrain 5 epochs)

In [6]:
import subprocess

ORIG_HIST = MARK1_DIR / "mark_4_outputs" / "mark_4_history.csv"
HIST_PATH = OUT["mark_4"] / "mark_4_history.csv"


def gpu_temperature():
    if not torch.cuda.is_available():
        return np.nan
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=temperature.gpu", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True)
        return float(result.stdout.splitlines()[0])
    except Exception:
        return np.nan


def freeze_batchnorm_running_stats(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


if REUSE_HISTORY and ORIG_HIST.is_file():
    shutil.copy2(ORIG_HIST, HIST_PATH)
    for best_csv in ("best_validation_patient_metrics.csv",
                     "best_validation_per_slice.csv", "best_validation_size_metrics.csv"):
        source = MARK1_DIR / "mark_4_outputs" / best_csv
        if source.is_file():
            shutil.copy2(source, OUT["mark_4"] / best_csv)
    history = json.loads(pd.read_csv(HIST_PATH).to_json(orient="records"))
    print(f"REUSE: Mark 4 history copied ({len(history)} epochs).")
else:
    history, best_score, best_epoch, best_result = [], -np.inf, None, None
    for epoch in range(1, 6):
        model.train()
        freeze_batchnorm_running_stats(model)
        train_loss, gradients_finite = 0.0, True
        for batch in train_loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            masks = batch["mask"].to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = loss_function(logits, masks)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite training loss.")
            loss.backward()
            gradients_finite &= all(p.grad is None or torch.isfinite(p.grad).all()
                                    for p in model.parameters())
            if not gradients_finite:
                raise FloatingPointError("Non-finite gradient.")
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += float(loss) * len(images)
        result, patient_metrics, slice_metrics, size_metrics = evaluate(model)
        scheduler.step()
        record = {"epoch": epoch, "train_loss": train_loss / len(train_dataset), **result,
                  "learning_rate": optimizer.param_groups[0]["lr"],
                  "end_temperature_c": gpu_temperature(), "gradients_finite": gradients_finite}
        history.append(record)
        pd.DataFrame(history).to_csv(HIST_PATH, index=False)
        torch.save({"epoch": epoch, "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(), "history": history,
                    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
                    "source_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256,
                    "roi_rule": {"liver_threshold": 0.50, "padding": 16,
                                 "component_mode": "largest_3d"},
                    "input_window_hu": list(BROAD_WINDOW)},
                   OUT["mark_4"] / "mark_4_last.pth")
        if result["mean_patient_dice"] > best_score:
            best_score = result["mean_patient_dice"]
            best_epoch = epoch
            best_result = result
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(), "history": history,
                        "manifest_sha256": EXPECTED_MANIFEST_SHA256,
                        "source_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256},
                       OUT["mark_4"] / "mark_4_best.pth")
            patient_metrics.to_csv(OUT["mark_4"] / "best_validation_patient_metrics.csv", index=False)
            slice_metrics.to_csv(OUT["mark_4"] / "best_validation_per_slice.csv", index=False)
            size_metrics.to_csv(OUT["mark_4"] / "best_validation_size_metrics.csv", index=False)
        print(f"epoch={epoch} train={record['train_loss']:.4f} "
              f"patient={record['mean_patient_dice']:.4f} "
              f"V104={record['volume_104_dice']:.4f} V116={record['volume_116_dice']:.4f} "
              f"Q1={record['q1_detected_pct']:.1f}% emptyFP={record['empty_slice_false_positive_pct']:.1f}%")
    print("REBUILD: 5-epoch smoke training completed.")

history_frame = pd.DataFrame(history)
display(history_frame)

REUSE: Mark 4 history copied (5 epochs).


,epoch,train_loss,validation_loss,mean_patient_dice,median_patient_dice,worst_patient_dice,volume_104_dice,volume_116_dice,q1_detected_pct,positive_predicted_empty_pct,empty_slice_false_positive_pct,learning_rate,start_temperature_c,end_temperature_c,elapsed_seconds,gradients_finite
0,1,0.130088,0.056866,0.297799,0.246682,0.001938,0.001938,0.004926,21.292776,50.287908,0.954060,0.000274,68.0,84.0,1068.012945,True
1,2,0.102449,0.062709,0.309006,0.210320,0.003376,0.003525,0.003376,40.304183,39.827255,3.318469,0.000207,74.0,86.0,1058.124801,True
2,3,0.087086,0.068398,0.305599,0.158734,0.004894,0.008801,0.004894,47.528517,33.397313,4.666598,0.000123,73.0,87.0,1045.736382,True
3,4,0.070738,0.057609,0.354879,0.320160,0.010146,0.010146,0.035249,41.064639,39.155470,2.530333,0.000056,74.0,85.0,1067.490074,True
4,5,0.060989,0.060651,0.363866,0.414723,0.010780,0.067187,0.010780,42.585551,36.852207,3.422172,0.000030,74.0,86.0,1037.105877,True


### 4.6 Smoke-test trajectories + best full-image predictions

In [7]:
figure, axes = plt.subplots(2, 3, figsize=(20, 11))
axes[0, 0].plot(history_frame["epoch"], history_frame["train_loss"], marker="o", label="Train")
axes[0, 0].set_title("Training loss"); axes[0, 0].legend()
axes[0, 1].plot(history_frame["epoch"], history_frame["mean_patient_dice"], marker="o")
axes[0, 1].axhline(CONTINUATION_TARGETS["mean_patient_dice"], linestyle="--", color="#444")
axes[0, 1].set_title("Mean patient Dice")
axes[0, 2].plot(history_frame["epoch"], history_frame["volume_104_dice"], marker="o", label="V104")
axes[0, 2].plot(history_frame["epoch"], history_frame["volume_116_dice"], marker="s", label="V116")
axes[0, 2].legend(); axes[0, 2].set_title("Focus-patient Dice")
axes[1, 0].plot(history_frame["epoch"], history_frame["q1_detected_pct"], marker="o")
axes[1, 0].axhline(CONTINUATION_TARGETS["q1_detected_pct"], linestyle="--", color="#444")
axes[1, 0].set_title("Q1 detection (%)")
axes[1, 1].plot(history_frame["epoch"], history_frame["positive_predicted_empty_pct"],
                marker="o", label="Positive empty")
axes[1, 1].plot(history_frame["epoch"], history_frame["empty_slice_false_positive_pct"],
                marker="s", label="Empty FP")
axes[1, 1].legend(); axes[1, 1].set_title("Slice error rates (%)")
if "end_temperature_c" in history_frame:
    axes[1, 2].plot(history_frame["epoch"], history_frame["end_temperature_c"],
                    marker="o", color="#F28E2B")
    axes[1, 2].axhline(84, linestyle="--", color="#444")
    axes[1, 2].set_title("GPU temperature (C)")
for axis in axes.flat:
    axis.set_xlabel("Epoch")
figure.suptitle("Mark 4 two-stage validation smoke", fontsize=18, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4"] / "mark_4_smoke_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

# Best-epoch full-image predictions (uses the frozen Mark 4 best checkpoint)
best_row = history_frame.loc[history_frame["mean_patient_dice"].idxmax()]
best_epoch = int(best_row["epoch"])
if REUSE_HISTORY:
    best_payload = torch.load(MARK1_DIR / "mark_4_outputs" / "mark_4_best.pth",
                              map_location="cpu", weights_only=False)
    model.load_state_dict(best_payload["model_state"], strict=True)
model.eval()
focus_rows = []
for volume_id in [104, 116, 108, 109]:
    candidates = validation_manifest.loc[
        validation_manifest["volume_id"].eq(volume_id)
        & validation_manifest["tumor_pixels"].gt(0)]
    focus_rows.append(candidates.nlargest(1, "tumor_pixels").iloc[0])

figure, axes = plt.subplots(4, 4, figsize=(15, 15))
for row_axes, row in zip(axes, focus_rows):
    dataset_index = int(validation_manifest.index[
        validation_manifest["sample_id"].eq(row.sample_id)][0])
    item = validation_dataset[dataset_index]
    with torch.inference_mode():
        probability_roi = torch.sigmoid(
            model(item["image"][None].to(DEVICE)))[0, 0].cpu().numpy()
    probability_full = probability_to_full(probability_roi, item["box"].numpy())
    prediction = probability_full >= 0.50
    with Image.open(DATASET_ROOT / row.image_path) as handle:
        image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
    with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
        truth = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
    panels = [(image, "Full CT"), (truth, "Expected tumor"),
              (probability_full, "Full probability"), (prediction, "Generated tumor")]
    for axis, (panel, title) in zip(row_axes, panels):
        axis.imshow(panel, cmap="magma" if "probability" in title else "gray", vmin=0, vmax=1)
        axis.set_title(title); axis.axis("off")
    row_axes[0].set_ylabel(f"V{int(row.volume_id)}", fontsize=10)
figure.suptitle(f"Best epoch {best_epoch} full-image predictions", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4"] / "best_full_image_predictions.png", dpi=170, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_26868\1434812655.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\alanm\AppData\Local\Temp\ipykernel_26868\1170923708.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  return {"image": torch.from_numpy(image_roi[None]).float(),
C:\Users\alanm\AppData\Local\Temp\ipykernel_26868\1434812655.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.7 Apply the continuation gate + write the Mark 4 gate

In [8]:
best_record = history_frame.loc[history_frame["mean_patient_dice"].idxmax()].to_dict()
passes = target_passes(best_record, CONTINUATION_TARGETS)
passes["gradients_finite"] = bool(best_record.get("gradients_finite", True))
continuation_passed = all(passes.values())
final_passes = target_passes(best_record, FINAL_TARGETS)

m4_gate = {
    "status": "mark_4_smoke_pass" if continuation_passed else "mark_4_smoke_fail",
    "best_epoch": int(best_record["epoch"]),
    "best_metrics": {key: float(best_record[key]) for key in CONTINUATION_TARGETS},
    "continuation_targets": CONTINUATION_TARGETS,
    "continuation_passes": passes,
    "continuation_gate_passed": continuation_passed,
    "final_validation_targets": FINAL_TARGETS,
    "final_targets_passed": final_passes,
    "all_final_targets_passed": all(final_passes.values()),
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "source_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256,
    "overfit_checkpoint_used_for_initialization": False,
    "test_images_accessed": False,
    "decision": ("PROCEED_TO_BOUNDED_LONGER_TWO_STAGE_VALIDATION_RUN" if continuation_passed
                 else "STOP_AND_DIAGNOSE_TWO_STAGE_SMOKE_FAILURE"),
    "next_notebook": ("mark_5_two_stage_bounded_continuation" if continuation_passed
                      else "mark_4_failure_diagnostics"),
}
(OUT["mark_4"] / "mark_4_gate_result.json").write_text(json.dumps(m4_gate, indent=2))
expected_actual = pd.DataFrame([
    {"metric": key, "actual": best_record[key],
     "continuation_target": CONTINUATION_TARGETS[key], "continuation_passed": passes[key],
     "final_target": FINAL_TARGETS[key], "final_passed": final_passes[key]}
    for key in CONTINUATION_TARGETS])
expected_actual.to_csv(OUT["mark_4"] / "expected_vs_actual.csv", index=False)
display(pd.DataFrame([m4_gate]).T.rename(columns={0: "value"}))
display(expected_actual)
print(m4_gate["decision"])

# ---- Reproduction check against the original gate ----
orig_m4 = json.loads((MARK1_DIR / "mark_4_outputs" / "mark_4_gate_result.json").read_text())
diffs = {k: abs(float(m4_gate["best_metrics"][k]) - float(orig_m4["best_metrics"][k]))
         for k in CONTINUATION_TARGETS}
print("Mark 4 reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 4 gate drifted from the original!"
assert m4_gate["status"] == orig_m4["status"] and m4_gate["best_epoch"] == orig_m4["best_epoch"]
print("PASS: Mark 4 gate matches the original mark_4_gate_result.json.")

,value
status,mark_4_smoke_fail
best_epoch,5
best_metrics,"{'mean_patient_dice': 0.3638655124, 'volume_10..."
continuation_targets,"{'mean_patient_dice': 0.3329, 'volume_104_dice..."
continuation_passes,"{'mean_patient_dice': True, 'volume_104_dice':..."
continuation_gate_passed,False
final_validation_targets,"{'mean_patient_dice': 0.406915, 'volume_104_di..."
final_targets_passed,"{'mean_patient_dice': False, 'volume_104_dice'..."
all_final_targets_passed,False
manifest_sha256,575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63...


,metric,actual,continuation_target,continuation_passed,final_target,final_passed
0,mean_patient_dice,0.363866,0.3329,True,0.406915,False
1,volume_104_dice,0.067187,0.0500,True,0.500000,False
2,volume_116_dice,0.010780,0.0100,True,0.050000,False
3,q1_detected_pct,42.585551,35.0000,True,45.000000,False
4,positive_predicted_empty_pct,36.852207,35.0000,False,20.000000,False
5,empty_slice_false_positive_pct,3.422172,20.0000,True,15.000000,True


STOP_AND_DIAGNOSE_TWO_STAGE_SMOKE_FAILURE
Mark 4 reproduction check: {'mean_patient_dice': 0.0, 'volume_104_dice': 0.0, 'volume_116_dice': 0.0, 'q1_detected_pct': 0.0, 'positive_predicted_empty_pct': 0.0, 'empty_slice_false_positive_pct': 0.0}
PASS: Mark 4 gate matches the original mark_4_gate_result.json.


In [9]:

# ---- Standard phase summary dashboard (centralized visualization) ----
render_summary_dashboard("mark_4")


PASS: mark_4_summary_dashboard.png -> D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\04_mark_4\figures


C:\Users\alanm\AppData\Local\Temp\ipykernel_26868\4137997672.py:384: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ---------------------------------------------------------------------------
# Publish key mark_4 artifacts to the shared/legacy folder other code reads.
#
# External consumers: part-2 steps 01/02/03/15 (`mark_4_best.pth`, `validation_roi_manifest.csv`).
# These code files hardcode artifact paths under `mark 1/mark_4_outputs/`, so
# after every run the freshly produced artifacts are mirrored there to keep
# those code files working. Values are recomputed from frozen inputs and
# verified against the original gates (reproduction check above), so the
# mirrored files are equivalent.
# ---------------------------------------------------------------------------
import shutil

PUBLISH_DIR = MARK1_DIR / "mark_4_outputs"
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

published = []
for pattern in ("*.csv", "*.json", "*.pth"):
    for source in sorted(OUT["mark_4"].glob(pattern)):
        shutil.copy2(source, PUBLISH_DIR / source.name)
        published.append(PUBLISH_DIR / source.name)

assert published, f"no mark_4 artifacts found to publish"
print(f"PUBLISHED {len(published)} mark_4 artifacts to {PUBLISH_DIR}:")
for artifact in sorted(published):
    print("  " + str(artifact))

PUBLISHED 8 mark_4 artifacts to D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4_outputs:
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4_outputs\best_validation_patient_metrics.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4_outputs\best_validation_per_slice.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4_outputs\best_validation_size_metrics.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4_outputs\broad_png_source_parity.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4_outputs\expected_vs_actual.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4_outputs\mark_4_gate_result.json
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4_outputs\mark_4_history.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4_outputs\validation_roi_manifest.csv
